In [ ]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 729.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 1.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.6.1
    Uninstalling fsspec-2024.6.1:
      Successfully uninstalled fsspec-2024.6.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.6.1 requires fsspec==2024.6.1, but you have fsspec 2024.9.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.4/145.4 kB 1.4 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.9.0
    Uninstalling fsspec-2024.9.0:
      Successfully uninstalled fsspec-2024.9.0
ERROR: pip's dependency resolver does not currently take into account all the packages that

In [6]:
import torch
import torch.nn.functional as F
from torch_geometric.loader import DataLoader  # Updated import for DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool  # Import GCNConv and pooling
from torch_geometric.datasets import TUDataset
from torch_geometric.transforms import NormalizeFeatures
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, log_loss
from sklearn.model_selection import train_test_split
from torch.nn import Dropout, GRU, Linear  # Corrected imports for GRU, Dropout, and Linear
from torch.optim.lr_scheduler import ReduceLROnPlateau

# DEFINE GRNN MODEL WITH DROPOUT AND MORE HIDDEN LAYERS
class GRNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=3, dropout_rate=0.3):
        super(GRNN, self).__init__()
        self.gcn_layers = torch.nn.ModuleList()
        self.gcn_layers.append(GCNConv(in_channels, hidden_channels))
        for _ in range(num_layers - 1):
            self.gcn_layers.append(GCNConv(hidden_channels, hidden_channels))

        self.dropout = Dropout(dropout_rate)
        self.gru = GRU(hidden_channels, hidden_channels, batch_first=True)
        self.linear = Linear(hidden_channels, out_channels)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        # Apply GCN layers with dropout
        for gcn in self.gcn_layers:
            x = gcn(x, edge_index).relu()
            x = self.dropout(x)

        # Apply global mean pooling
        x = global_mean_pool(x, batch)

        # Reshape for GRU: (batch_size, sequence_length, feature_dim)
        x = x.unsqueeze(0)

        # Apply GRU
        x, _ = self.gru(x)
        x = x.squeeze(0)

        # Final linear layer
        x = self.linear(x)

        return F.log_softmax(x, dim=1)

# Train the model
def train_model(model, loader, epochs=200, lr=0.001, weight_decay=1e-4):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10, min_lr=1e-6)
    criterion = torch.nn.CrossEntropyLoss()

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for data in loader:
            optimizer.zero_grad()
            out = model(data)
            loss = criterion(out, data.y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step(total_loss / len(loader))

    return model

# EVALUATE MODEL FUNCTION
def evaluate_model(model, loader, num_classes):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for data in loader:
            out = model(data)
            preds = out.argmax(dim=1)

            if data.y.ndim == 2:
                labels = data.y.argmax(dim=1)
            else:
                labels = data.y

            all_preds.append(out)
            all_labels.append(labels)

    all_preds = torch.cat(all_preds, dim=0)
    all_labels = torch.cat(all_labels, dim=0).view(-1)

    out_prob = F.softmax(all_preds, dim=1).cpu()

    current_num_classes = out_prob.shape[1]
    if current_num_classes != num_classes:
        if current_num_classes < num_classes:
            padding = num_classes - current_num_classes
            out_prob = torch.cat([out_prob, torch.zeros(out_prob.size(0), padding)], dim=1)
        else:
            out_prob = out_prob[:, :num_classes]

    auc_roc = roc_auc_score(all_labels.cpu().numpy(), out_prob[:, 1], average='weighted') if num_classes == 2 else roc_auc_score(all_labels.cpu().numpy(), out_prob, multi_class='ovr', average='weighted')
    logloss = log_loss(all_labels.cpu().numpy(), out_prob)

    test_acc = (all_preds.argmax(dim=1) == all_labels).sum().item() / len(all_labels)
    f1 = f1_score(all_labels.cpu(), all_preds.argmax(dim=1).cpu(), average='weighted')
    precision = precision_score(all_labels.cpu(), all_preds.argmax(dim=1).cpu(), average='weighted')
    recall = recall_score(all_labels.cpu(), all_preds.argmax(dim=1).cpu(), average='weighted')

    return {
        'accuracy': test_acc,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'auc_roc': auc_roc,
        'log_loss': logloss
    }

# LOAD AND AUGMENT DATASETS
dataset_nci1 = TUDataset(root='/content/NCI1', name='NCI1', transform=NormalizeFeatures())
dataset_mutag = TUDataset(root='/content/MUTAG', name='MUTAG', transform=NormalizeFeatures())

# SPLIT DATASETS INTO TRAINING AND TESTING SETS (70/30)
train_idx_nci1, test_idx_nci1 = train_test_split(range(len(dataset_nci1)), test_size=0.3, random_state=42)
train_dataset_nci1 = dataset_nci1[train_idx_nci1]
test_dataset_nci1 = dataset_nci1[test_idx_nci1]

train_idx_mutag, test_idx_mutag = train_test_split(range(len(dataset_mutag)), test_size=0.3, random_state=42)
train_dataset_mutag = dataset_mutag[train_idx_mutag]
test_dataset_mutag = dataset_mutag[test_idx_mutag]

train_loader_nci1 = DataLoader(train_dataset_nci1, batch_size=32, shuffle=True)
test_loader_nci1 = DataLoader(test_dataset_nci1, batch_size=32, shuffle=False)

train_loader_mutag = DataLoader(train_dataset_mutag, batch_size=32, shuffle=True)
test_loader_mutag = DataLoader(test_dataset_mutag, batch_size=32, shuffle=False)

# INITIALIZE AND TRAIN MODEL ON NCI1
model_nci1 = GRNN(in_channels=dataset_nci1.num_node_features, hidden_channels=64, out_channels=dataset_nci1.num_classes, dropout_rate=0.5)
train_model(model_nci1, train_loader_nci1, epochs=200)

# EVALUATE ON NCI1 DATASET
print("\nResults on NCI1 Dataset Before Transfer Learning:")
nci1_results = evaluate_model(model_nci1, test_loader_nci1, num_classes=dataset_nci1.num_classes)
print(nci1_results)

# TRANSFER LEARNING TO MUTAG DATASET
model_mutag = GRNN(in_channels=dataset_mutag.num_node_features, hidden_channels=64, out_channels=dataset_mutag.num_classes, dropout_rate=0.5)
state_dict_nci1 = model_nci1.state_dict()
state_dict_mutag = model_mutag.state_dict()

for key in state_dict_nci1:
    if key in state_dict_mutag and state_dict_nci1[key].shape == state_dict_mutag[key].shape:
        state_dict_mutag[key] = state_dict_nci1[key]

model_mutag.load_state_dict(state_dict_mutag, strict=False)
train_model(model_mutag, train_loader_mutag, epochs=200)

# EVALUATE ON MUTAG DATASET
print("\nResults on MUTAG Dataset After Transfer Learning:")
mutag_results = evaluate_model(model_mutag, test_loader_mutag, num_classes=dataset_mutag.num_classes)
print(mutag_results)



Results on NCI1 Dataset Before Transfer Learning:


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:2922: UserWarning: The y_pred values do not sum to one. Starting from 1.5 thiswill result in an error.
  warnings.warn(


{'accuracy': 0.7023519870235199, 'f1': 0.7021796682308231, 'precision': 0.7048105740130117, 'recall': 0.7023519870235199, 'auc_roc': 0.7705376989770656, 'log_loss': 0.5837534539492009}

Results on MUTAG Dataset After Transfer Learning:
{'accuracy': 0.8070175438596491, 'f1': 0.7764872856298048, 'precision': 0.8494736842105264, 'recall': 0.8070175438596491, 'auc_roc': 0.8518518518518519, 'log_loss': 0.47832033392678}


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:2922: UserWarning: The y_pred values do not sum to one. Starting from 1.5 thiswill result in an error.
  warnings.warn(
